# Machine Learning Lab: Learning a Step Function
## TV advertising budget → product sales

**Learning goals:** Understand step functions, train a Decision Tree Regressor, interpret the learned thresholds, and explore underfitting and overfitting.

**How to use:** Open this notebook in Google Colab and select **Runtime → Run all**. You can change the parameters marked **✏️ TRY THIS** and rerun the relevant cells. No prior coding experience is required.

> **Important:** The data in this notebook are **synthetic teaching data**, not real observations. We construct them using the example function on the lecture slide and add random variation. The model does not receive the original thresholds as inputs; it must estimate them from the data.


## 1. The hypothesis: Four constant sales levels

The example assumes the expected number of units sold changes at certain advertising-budget thresholds:

$$
f(x)=\begin{cases}
8{,}500 & x<75{,}000\\
12{,}600 & 75{,}000\leq x<150{,}000\\
16{,}700 & 150{,}000\leq x<225{,}000\\
18{,}700 & x\geq225{,}000.
\end{cases}
$$

Here **x = TV advertising costs (€)** and **f(x) = expected sales (units)**.

A step function is *constant inside each interval* and changes *abruptly at a threshold*. A decision tree can represent such a function by splitting the x-axis into intervals.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

def true_sales(budgets):
    """Lecture example: expected sales before random noise."""
    x = np.asarray(budgets)
    return np.select(
        [x < 75_000, x < 150_000, x < 225_000],
        [8_500, 12_600, 16_700],
        default=18_700
    )

budget_grid = np.linspace(0, 300_000, 1001)
plt.figure(figsize=(10, 4))
plt.step(budget_grid, true_sales(budget_grid), where='post', linewidth=2.5,
         label='Lecture example (true step function)')
plt.xlabel('TV advertising costs (€)')
plt.ylabel('Expected sales (units)')
plt.title('The underlying step function')
plt.xticks(np.arange(0, 300_001, 50_000))
plt.grid(alpha=.25)
plt.legend()
plt.show()

## 2. Create a small dataset

Suppose we observe **120 advertising campaigns** with different TV budgets. Actual sales fluctuate because other factors also affect sales. We therefore add random noise to the expected values.

**✏️ TRY THIS:** Change `noise_standard_deviation` from `700` to `0` or `1500`. How does that change the data and the model's ability to detect boundaries?


In [ ]:
number_of_campaigns = 120               # ✏️ TRY THIS
noise_standard_deviation = 700             # ✏️ TRY THIS: 0, 700 or 1500

rng = np.random.default_rng(RANDOM_SEED)
budgets = rng.uniform(0, 300_000, number_of_campaigns)
observed_sales = true_sales(budgets) + rng.normal(
    loc=0, scale=noise_standard_deviation, size=number_of_campaigns
)
observed_sales = np.maximum(0, observed_sales)

data = pd.DataFrame({
    'TV_budget_EUR': budgets.round(0).astype(int),
    'Sales_units': observed_sales.round(0).astype(int)
}).sort_values('TV_budget_EUR').reset_index(drop=True)
display(data.head(10))

plt.figure(figsize=(10, 4))
plt.scatter(data['TV_budget_EUR'], data['Sales_units'], alpha=.55,
            label='Observed sales (synthetic)')
plt.step(budget_grid, true_sales(budget_grid), where='post', color='black',
         linewidth=2, label='Underlying step function')
plt.xlabel('TV advertising costs (€)')
plt.ylabel('Sales (units)')
plt.title('What do you think the relationship looks like?')
plt.grid(alpha=.25)
plt.legend()
plt.show()

### Think before training

1. Is a straight line an adequate hypothesis for this dataset?
2. At approximately which advertising budgets does the sales level change?
3. Are the step boundaries obvious from noisy observations? Why or why not?


## 3. Split data into training and test sets

We use **80%** of the campaigns to train the model and reserve **20%** for an evaluation on unseen observations. The input feature is the TV budget and the target variable is sales.


In [ ]:
X = data[['TV_budget_EUR']]  # Feature: TV advertising budget
y = data['Sales_units']       # Target: product sales

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED
)
print(f'Training examples: {len(X_train)}')
print(f'Test examples:     {len(X_test)}')

## 4. Train a model that learns steps

A **Decision Tree Regressor** repeatedly chooses boundaries that reduce prediction error. Each leaf predicts a constant sales value, producing a step-shaped function.

**✏️ TRY THIS:** `max_leaf_nodes=4` allows at most four constant regions. Set it to **2**, **4**, or **20** and compare results.

> The model does **not** know the exact €75k / €150k / €225k thresholds. It estimates split points from the observations, so its fitted thresholds and sales levels will differ from the slide.


In [ ]:
max_leaf_nodes = 4    # ✏️ TRY THIS: 2, 4 or 20

model = DecisionTreeRegressor(
    max_leaf_nodes=max_leaf_nodes,
    min_samples_leaf=3,
    random_state=RANDOM_SEED
)
model.fit(X_train, y_train)

x_plot = pd.DataFrame({'TV_budget_EUR': budget_grid})
predicted_curve = model.predict(x_plot)

plt.figure(figsize=(11, 5))
plt.scatter(X_train['TV_budget_EUR'], y_train, alpha=.35, label='Training observations')
plt.step(budget_grid, true_sales(budget_grid), where='post', color='black',
         linewidth=2, linestyle='--', label='Underlying lecture function')
plt.step(budget_grid, predicted_curve, where='post', linewidth=2.5,
         label=f'Learned decision tree ({model.get_n_leaves()} leaves)')
plt.xlabel('TV advertising costs (€)')
plt.ylabel('Sales (units)')
plt.title('The learned step function vs. the underlying function')
plt.grid(alpha=.25)
plt.legend()
plt.show()

print('Learned decision rules:')
print(export_text(model, feature_names=['TV_budget_EUR']))

## 5. Evaluate on unseen data

We compare predictions with actual test-set observations. **MAE** is the average absolute prediction error in units sold; **RMSE** penalizes larger errors more heavily; **R²** summarizes explanatory performance (higher is generally better, but it can be negative).


In [ ]:
train_predictions = model.predict(X_train)
test_predictions = model.predict(X_test)

metrics = pd.DataFrame({
    'Dataset': ['Training', 'Test'],
    'MAE (units)': [mean_absolute_error(y_train, train_predictions),
                    mean_absolute_error(y_test, test_predictions)],
    'RMSE (units)': [np.sqrt(mean_squared_error(y_train, train_predictions)),
                     np.sqrt(mean_squared_error(y_test, test_predictions))],
    'R²': [r2_score(y_train, train_predictions), r2_score(y_test, test_predictions)]
}).round(2)
display(metrics)

plt.figure(figsize=(7, 5))
plt.scatter(y_test, test_predictions, alpha=.75)
lims = [min(y_test.min(), test_predictions.min()) - 500,
        max(y_test.max(), test_predictions.max()) + 500]
plt.plot(lims, lims, 'k--', label='Perfect prediction')
plt.xlim(lims)
plt.ylim(lims)
plt.xlabel('Observed sales (test set)')
plt.ylabel('Predicted sales (test set)')
plt.title('How accurate are the predictions?')
plt.grid(alpha=.25)
plt.legend()
plt.show()

## 6. Make predictions for advertising budgets

**✏️ TRY THIS:** Change the budgets below. Predict the results before running the cell. Compare the learned model's values with the underlying lecture function.


In [ ]:
example_budgets = [50_000, 100_000, 175_000, 250_000]  # ✏️ TRY THIS
examples = pd.DataFrame({'TV_budget_EUR': example_budgets})
examples['Model_prediction_units'] = model.predict(examples[['TV_budget_EUR']]).round(0).astype(int)
examples['Lecture_function_units'] = true_sales(examples['TV_budget_EUR'])
display(examples)

## 7. Explore underfitting and overfitting

We train trees with different maximum numbers of leaves on **the same training data** and evaluate them on **the same held-out test set**.

- **Too few leaves (e.g., 2):** The function may be too simple to capture the true four-level pattern (**underfitting**).
- **Four leaves:** Matches the complexity of the underlying example, though the data may still lead to imperfect thresholds.
- **Many leaves (e.g., 20 or more):** The tree can fit random fluctuations and may generalize less well (**overfitting**).

*These are typical tendencies, not guarantees for every random dataset.*


In [ ]:
leaf_options = [2, 4, 8, 20, 40]
results = []

plt.figure(figsize=(11, 5))
plt.scatter(X_train['TV_budget_EUR'], y_train, alpha=.20, label='Training data')
plt.step(budget_grid, true_sales(budget_grid), where='post', color='black',
         linestyle='--', linewidth=2, label='Underlying function')

for leaves in leaf_options:
    candidate = DecisionTreeRegressor(
        max_leaf_nodes=leaves, min_samples_leaf=2, random_state=RANDOM_SEED
    )
    candidate.fit(X_train, y_train)
    plt.step(budget_grid, candidate.predict(x_plot), where='post',
             alpha=.75, label=f'Max {leaves} leaves')
    results.append({
        'Max leaves': leaves,
        'Actual leaves': candidate.get_n_leaves(),
        'Training MAE': mean_absolute_error(y_train, candidate.predict(X_train)),
        'Test MAE': mean_absolute_error(y_test, candidate.predict(X_test))
    })
plt.xlabel('TV advertising costs (€)')
plt.ylabel('Sales (units)')
plt.title('Comparing model complexity')
plt.grid(alpha=.25)
plt.legend(ncol=2)
plt.show()

comparison = pd.DataFrame(results).round(1)
display(comparison)

plt.figure(figsize=(8, 4))
plt.plot(comparison['Max leaves'], comparison['Training MAE'], marker='o', label='Training MAE')
plt.plot(comparison['Max leaves'], comparison['Test MAE'], marker='o', label='Test MAE')
plt.xlabel('Maximum number of leaves')
plt.ylabel('Mean absolute error (units)')
plt.title('Training error vs. generalization error')
plt.grid(alpha=.25)
plt.legend()
plt.show()

## 8. Inspect the fitted decision tree (optional)

Each internal node asks a question about the TV budget (e.g., "Is budget ≤ threshold?"). Each leaf gives a constant predicted number of units sold.


In [ ]:
plt.figure(figsize=(15, 6))
plot_tree(model, feature_names=['TV budget (€)'],
          filled=True, rounded=True, fontsize=9)
plt.title('How the trained decision tree creates a step function')
plt.show()

## 9. Student exercise (10–15 minutes)

**Task A — Understand the hypothesis space**

1. Explain why a four-leaf regression tree can represent the example's four constant sales levels.
2. Does the algorithm estimate the step boundaries, the sales values, or both?
3. Why do the learned boundary locations not necessarily equal €75,000, €150,000 and €225,000?

**Task B — Experiment**

4. Train models with `max_leaf_nodes = 2, 4, 20`. Record which one has the lowest **test MAE**.
5. Set `noise_standard_deviation = 0` in Section 2, then rerun Sections 2–7. What changes?
6. Set `noise_standard_deviation = 1500`, rerun, and explain the difference between fitting training data and predicting new cases.

**Task C — Reflect**

7. If sales rise smoothly instead of jumping at sharp boundaries, would a step-function hypothesis still be sensible? What are its limitations?
8. Would you choose the most complex tree just because it has the smallest *training* error? Explain.

**Takeaway:** A decision tree is one way to *learn a step-shaped regression function* from data. Restricting the number of leaves controls the size of the hypothesis space and changes the trade-off between underfitting and overfitting.
